### TOOLS

#### Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Hmm, let\'s start by thinking about what I know about parrots. They\'re birds, right? Some of them can mimic human speech. But why do they do that? Is it just to imitate, or is there a reason behind it?\n\nFirst, maybe they have some kind of social need. Like, if they live with humans, they might learn to talk to communicate or interact. But wait, in the wild, do parrots talk? Probably not. So maybe it\'s more about their environment. If they\'re kept in captivity, they pick up words from their owners. But why not other sounds? Maybe their vocal cords are structured in a way that allows for mimicking a variety of sounds, including human speech.\n\nAlso, maybe it\'s related to their intelligence. Parrots are known to be smart. They can solve problems and have complex social structures. In the wild, they might use mimicry as a way to communicate with other parrots or to scare predators. But in captivity, they mig

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")

print(response)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'd7mz3012v', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.146830855, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.006244913, 'prompt_tokens_details': None, 'queue_time': 0.162554637, 'total_time': 0.153075768}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f412d-5f35-7e61-a

### Tools Execution Loops

In [5]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect result
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

# "The current weather in Boston is 72°F and sunny."

The current weather in Boston is sunny.
